In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
# from mediapipe import solutions
import numpy as np
import cv2
import pandas as pd
import math
import os
import json
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
cwd = os.getcwd()
base_data_path = os.path.join(cwd, "asl citizen data", "ASL_Citizen")
videos_path = os.path.join(base_data_path, "videos")
splits_path = os.path.join(base_data_path, "splits")

In [ ]:
train = pd.read_csv(os.path.join(splits_path, "train.csv"))
test = pd.read_csv(os.path.join(splits_path, "test.csv"))
val = pd.read_csv(os.path.join(splits_path, "val.csv"))

In [ ]:
train.rename(columns={"Video file": "path", "Participant ID": "participant", "Gloss": "gloss", "ASL-LEX Code": "code"}, inplace=True)
test.rename(columns={"Video file": "path", "Participant ID": "participant", "Gloss": "gloss", "ASL-LEX Code": "code"}, inplace=True)
val.rename(columns={"Video file": "path", "Participant ID": "participant", "Gloss": "gloss", "ASL-LEX Code": "code"}, inplace=True)
train.head()

## Deprecated generator

The inline generator previously stored absolute landmark positions in relative-motion history and passed BGR frames to MediaPipe as RGB. It has been retired. Use the shared `DataGeneration` workflow in the next cell.


In [ ]:
# Generate corrected raw landmark data for every split.
# This is expensive and only needs to run when rebuilding the dataset.
from data_generation import DataGeneration

data_gen = DataGeneration(videos_path)

print("Gathering training data...")
data_gen.generate_from_dataset(train, "train", limit_count=False)
print("Gathering testing data...")
data_gen.generate_from_dataset(test, "test", limit_count=False)
print("Gathering validation data...")
data_gen.generate_from_dataset(val, "val", limit_count=False)


In [ ]:
# DATA CLEANING. SHOULD ONLY BE RUN WHEN GENERATING NEW DATA

import json


raw_gesture_data_folder = os.path.join(cwd, "asl_citizen_gesture_data/raw_data")
raw_test = os.path.join(raw_gesture_data_folder, "test")
raw_train = os.path.join(raw_gesture_data_folder, "train")
raw_val = os.path.join(raw_gesture_data_folder, "val")

def find_max_length(test_path, train_path, val_path):
    max_length = 0

    for path in [test_path, train_path, val_path]:
        for filename in os.listdir(path):
            file_path = os.path.join(path, filename)
            df = pd.read_csv(file_path)
            # print(filename, len(df))
            if len(df) > max_length:
                max_length = len(df)

    return max_length

def generate(raw_data_path, main_csv: pd.DataFrame, max_length, out_path=None):
    if out_path is None:
        print("You must specify an output path for the cleaned data.")
        return
    
    
    min_gesture_frame_length = 1
    # max_length = 0

    bad_files = []

    for filename in os.listdir(raw_data_path):
        file_path = os.path.join(raw_data_path, filename)
        
        df = pd.read_csv(file_path)
        
        if len(df) < min_gesture_frame_length:
            bad_files.append((file_path, filename))
            # print(i, file_path)
        
            
    # for path, name, i in bad_files:
    #     os.remove(path)

    # out_df = pd.DataFrame()
    out = {}

    for filename in os.listdir(raw_data_path):
        if any(filename == bad_name for _, bad_name in bad_files):
            continue
        
        file_path = os.path.join(raw_data_path, filename)
        df = pd.read_csv(file_path)
        if len(df) > max_length:
            df = df.iloc[:max_length].copy()
        
        if len(df) == 0:
            print(f"{file_path} is empty, skipping")
            continue
        
        if len(df) < max_length:
            current_frame = len(df) - 1 # NOTE: Previously did not subtract 1
            for i in range(max_length - len(df)):
                
                # frame 3, right hand present, left hand present, 126 zeros
                df.loc[len(df)] = [current_frame+1, 0, 0] + np.zeros(len(df.columns)-3).tolist()
                # df.loc[len(df)] = [current_frame+1, 1] + np.zeros(len(df.columns)-2).tolist()
            # print(f'i: {(i-1)*2}')
        
        df = df.reset_index(drop=True)
        
        clip_name = os.path.splitext(filename)[0]
        match = main_csv.loc[main_csv['path'] == os.path.splitext(filename)[0], 'gloss']
        
        if match.empty:
            print("No label found for file: ", clip_name)
            continue
        
        word = match.iloc[0]
        
        # new_df = pd.DataFrame([df.to_numpy().flatten()], columns=[f"row{index}_{col}" for index in df.index for col in df.columns])
        # new_df.insert(0, 'word', word)
        
        # out_df = pd.concat([out_df, new_df], ignore_index=True)
        out[clip_name] = {
            "word": word,
            "data": [df.values.tolist()],
        }
    
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    # out_df.to_csv(out_path, index=False)
    # return out_df

detected_max_length = find_max_length(raw_test, raw_train, raw_val)
max_length = 93
print(f'Detected maximum: {detected_max_length}; using fixed length: {max_length}')
generate(raw_train, train, max_length=max_length, out_path="asl_citizen_gesture_data/cleaned_train_data.json")
generate(raw_test, test, max_length=max_length, out_path="asl_citizen_gesture_data/cleaned_test_data.json")
generate(raw_val, val, max_length=max_length, out_path="asl_citizen_gesture_data/cleaned_val_data.json")

In [ ]:
# more data cleaning

train_data = json.load(open("asl_citizen_gesture_data/cleaned_train_data.json"))
test_data = json.load(open("asl_citizen_gesture_data/cleaned_test_data.json"))
val_data = json.load(open("asl_citizen_gesture_data/cleaned_val_data.json"))

def convert(input_data):
    out = []
    for key in input_data:
        word = input_data[key]["word"]
        data = input_data[key]["data"]
        out.append({
            "word": word,
            "data": data,
        })
    
    return out

new_train_data = convert(train_data)
new_test_data = convert(test_data)
new_val_data = convert(val_data)

with open("asl_citizen_gesture_data/cleaned_train_data.json", "w") as f:
    json.dump(new_train_data, f, indent=2)
with open("asl_citizen_gesture_data/cleaned_test_data.json", "w") as f:
    json.dump(new_test_data, f, indent=2)
with open("asl_citizen_gesture_data/cleaned_val_data.json", "w") as f:
    json.dump(new_val_data, f, indent=2)

In [ ]:
import tensorflow as tf

In [ ]:
# for new train

import json

train_data = json.load(open("asl_citizen_gesture_data/cleaned_train_data.json"))
test_data = json.load(open("asl_citizen_gesture_data/cleaned_test_data.json"))
val_data = json.load(open("asl_citizen_gesture_data/cleaned_val_data.json"))

combined_data = train_data + test_data + val_data
total_words = [word['word'] for word in combined_data]
unique_words = set(total_words)
unique_words = sorted(unique_words)
numbered_unique_words = {word:i for i, word in enumerate(unique_words)}
word_to_number = {i:word for word, i in numbered_unique_words.items()}

train_X = np.array([word['data'][0] for word in train_data])
train_y = np.array([numbered_unique_words[word['word']] for word in train_data])

test_X = np.array([word['data'][0] for word in test_data])
test_Y = np.array([numbered_unique_words[word['word']] for word in test_data])

val_X = np.array([word['data'][0] for word in val_data])
val_Y = np.array([numbered_unique_words[word['word']] for word in val_data])

print(train_X.shape)
print(numbered_unique_words)

In [ ]:
def add_velocity(X):
    velocity = np.diff(X, axis=1, prepend=X[:, :1, :])  # frame-to-frame delta
    return np.concatenate([X, velocity], axis=-1)  # doubles feature size


train_X = add_velocity(train_X)
val_X   = add_velocity(val_X)
test_X = add_velocity(test_X)

In [ ]:
# save example data to compare to live data

gesture_example_index = np.argwhere(val_Y == 2)[0]
example = val_X[gesture_example_index]
example = np.insert(example, 0, val_Y[gesture_example_index])
example_gesture = pd.DataFrame(example.flatten().reshape(1, -1))
# example_gesture['word'] = val_Y[gesture_example_index]

example_gesture.to_csv("asl_citizen_gesture_data/example_for_comparison.csv", index=False)

In [ ]:
parameters = {
    "word_to_number": word_to_number,
    "numbered_unique_words": numbered_unique_words,
    "frames_per_gesture": int(train_X.shape[1]),
    "data_per_frame": int(train_X.shape[2]),
    "feature_schema_version": 2,
    "motion_uses_timestamps": True,
    "hand_order": ["right", "left"],
    "model_file": "hand_gesture_modelv10.keras",
}

with open("parameters.json", "w") as file:
    json.dump(parameters, file, indent=2)


In [ ]:
def transformer_block(x, heads=4, ff_dim=128):
    # Self-attention with residual
    attn_output = tf.keras.layers.MultiHeadAttention(num_heads=heads, key_dim=32)(x, x)
    attn_output = tf.keras.layers.Dropout(0.1)(attn_output)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_output)  # residual
    
    # Feed-forward with residual
    ff = tf.keras.layers.Dense(ff_dim, activation="relu")(x)
    ff = tf.keras.layers.Dense(x.shape[-1])(ff)
    ff = tf.keras.layers.Dropout(0.1)(ff)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)  # residual
    return x

# --- Build model using Functional API ---
inputs = tf.keras.layers.Input(shape=train_X.shape[1:])

x = tf.keras.layers.Masking(mask_value=0.0)(inputs)

# BiLSTM layers
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True))(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)  # return_sequences=True to feed transformer
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Transformer block
x = transformer_block(x, heads=4, ff_dim=128)

# Pool across time dimension
x = tf.keras.layers.GlobalAveragePooling1D()(x)

# Classifier head
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(numbered_unique_words), activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
history = model.fit(train_X, train_y, epochs=150, validation_data=(val_X, val_Y), callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])

In [ ]:
# for old train (v1-v6)

train_data = pd.read_csv("asl_citizen_gesture_data/cleaned_train_data.csv")
test_data = pd.read_csv("asl_citizen_gesture_data/cleaned_test_data.csv")
val_data = pd.read_csv("asl_citizen_gesture_data/cleaned_val_data.csv")

# print(train_data.dtypes)
# print(val_data.dtypes)

total_words = train_data['word'] + test_data['word'] + val_data['word']
unique_words = set(train_data['word'])
numbered_unique_words = {word:i for i, word in enumerate(unique_words)}
word_to_number = {i:word for word, i in numbered_unique_words.items()}
train_data['word'] = train_data['word'].map(numbered_unique_words)
test_data['word'] = test_data['word'].map(numbered_unique_words)
val_data['word'] = val_data['word'].map(numbered_unique_words)

test_train = pd.DataFrame()

# for col in train_data.columns:
#     if not ("frame_index" in col or "hand_index" in col):
#         test_train[col] = train_data[col]
    

v2train = train_data.loc[:, ~train_data.columns.str.contains("frame_index|hand_index")]
v2test = test_data.loc[:, ~test_data.columns.str.contains("frame_index|hand_index")]
v2val = val_data.loc[:, ~val_data.columns.str.contains("frame_index|hand_index")]

print(word_to_number)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=train_X.shape[1:]),
    tf.keras.layers.Masking(mask_value=0.0),
    # tf.keras.layers.Normalization(),
    # tf.keras.layers.GaussianNoise(0.05),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    transformer_block(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),
    tf.keras.layers.Dropout(0.2),
    # tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(len(numbered_unique_words), activation="softmax")
])

lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts( initial_learning_rate=0.001, first_decay_steps=100, t_mul=2.0, m_mul=0.9 )
optimizer = tf.keras.optimizers.Adam(lr_schedule, clipnorm=1.0)
model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
# model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True,clipnorm=1.0), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
history = model.fit(train_X, train_y, epochs=150, validation_data=(val_X, val_Y), callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])

In [ ]:
# old training code (v1-v6)

# model = tf.keras.models.Sequential([
#     # tf.keras.layers.Input(shape=(df.shape[0], 3150)),
#     tf.keras.layers.Flatten(input_shape=(train_data.iloc[:, 1:].shape[1],)),
#     tf.keras.layers.Dense(64, activation="relu"),
#     # tf.keras.layers.Dense(1024, activation="relu"),
#     # tf.keras.layers.Dense(64, activation="relu"),
#     tf.keras.layers.Dense(32, activation="relu"),
#     tf.keras.layers.Dropout(0.2),
#     tf.keras.layers.Dense(len(numbered_unique_words), activation='softmax')
# ])

# model.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
# history = model.fit(train_data.iloc[:, 1:], train_data['word'], epochs=70, validation_data=(val_data.iloc[:, 1:], val_data['word']))

frames = 186
dpf = v2train.iloc[:, 1:].shape[1] // frames # train_data.iloc[:, 1:].shape[1] // frames

X_train = v2train.iloc[:, 1:].to_numpy(dtype=np.float32)
y_train = v2train["word"].to_numpy(dtype=np.int32)

X_val = v2val.iloc[:, 1:].to_numpy(dtype=np.float32)
y_val = v2val["word"].to_numpy(dtype=np.int32)

X_train = X_train.reshape(-1, frames, dpf)
X_val = X_val.reshape(-1, frames, dpf)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(frames, dpf)),
    tf.keras.layers.Normalization(),
    # tf.keras.layers.GaussianNoise(0.05),
    tf.keras.layers.Masking(mask_value=0.0),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=False)),
    # tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(len(numbered_unique_words), activation="softmax")
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=150, validation_data=(X_val, y_val), callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])

In [ ]:
model.evaluate(test_X, test_Y)

In [ ]:
from tensorflow.keras.models import load_model

# model = load_model("models/hand_gesture_modelv8.keras")

# X_test = v2test.iloc[:, 1:].to_numpy(dtype=np.float32)
# y_test = v2test["word"].to_numpy(dtype=np.int32)

# X_test = X_test.reshape(-1, frames, dpf)

model.evaluate(val_X, val_Y)
test_value_y_index = np.argwhere(val_Y == 4)[2]
example = val_X[test_value_y_index]
print(test_value_y_index, example.shape)

pred = model.predict(example)
print(np.argmax(pred))
print(pred)


In [ ]:
import matplotlib.pyplot as plt

# print(history.history.keys()) # View available metrics, e.g., ['loss', 'accuracy', 'val_loss', 'val_accuracy']

# 4. Plot training and validation loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.ylabel('Loss value')
plt.xlabel('Epoch')
plt.legend()

# 5. Plot training and validation accuracy (if applicable)
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.ylabel('Accuracy value')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
version = 10

model.save(f"models/hand_gesture_modelv{version}.keras")
model.save(f"website/models/hand_gesture_modelv{version}.keras")

# Activate v10 for the website only after both model saves succeed.
with open("parameters.json", "r") as source:
    saved_parameters = json.load(source)
with open("website/parameters.json", "w") as destination:
    json.dump(saved_parameters, destination, indent=2)


In [ ]:
print(find_max_length(raw_test, raw_train, raw_val))

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(f"models/hand_gesture_modelv9.keras")

probabilities = model.predict(test_X)
predictions = np.argmax(probabilities, axis=1)

print(
    classification_report(
        test_Y,
        predictions,
        target_names=[
            word_to_number[i]
            for i in range(len(word_to_number))
        ],
    )
)

print(confusion_matrix(test_Y, predictions))